# Test the solver methods

This notebook only imports and runs code from `cvrp/` and `solver/`. It does not define solver logic.

In [1]:
# Setup: let the notebook import from the project folder (one level up)
import os, sys
ROOT = os.path.abspath('..')
sys.path.insert(0, ROOT)

from cvrp.io import read_instance, format_solution
from cvrp.validator import validate
from cvrp.construct import nearest_neighbor, clarke_wright

## 1. Load the spec example

In [2]:
path = os.path.join(ROOT, 'instances', 'small', 'spec_example.txt')
demands, dist, capacity, vehicles = read_instance(path)

print('demands :', demands)
print('capacity:', capacity, ' vehicles:', vehicles)
for row in dist:
    print(row)

demands : [0, 2, 2, 3]
capacity: 5  vehicles: 2
[0.0, 2.5, 5.6, 6.0]
[2.5, 0.0, 3.9, 6.5]
[5.6, 3.9, 0.0, 5.2]
[6.0, 6.5, 5.2, 0.0]


## 2. Run Nearest Neighbor

In [3]:
routes = nearest_neighbor(demands, dist, capacity)
print(format_solution(routes, dist))

24.00
0 1 2 0
0 3 0



## 3. Check feasibility and compare with the known best (21.8)

In [4]:
errors, cost = validate(demands, dist, capacity, vehicles, routes)
print('feasible:', not errors)
for e in errors:
    print('  ', e)

best_known = 21.8
print('cost: %.2f   best known: %.2f   gap: %.1f%%' % (cost, best_known, 100 * (cost - best_known) / best_known))

feasible: True
cost: 24.00   best known: 21.80   gap: 10.1%


## 4. Route details (load and distance per route)

In [5]:
from cvrp.io import route_cost

for k, r in enumerate(routes, 1):
    load = sum(demands[v] for v in r)
    print('route %d: %-15s load %d/%d   distance %.2f' % (k, r, load, capacity, route_cost(r, dist)))

route 1: [0, 1, 2, 0]    load 4/5   distance 12.00
route 2: [0, 3, 0]       load 3/5   distance 12.00


## 5. Generated instances

Create random instances in memory with `generate()` from `generator/gen.py` (no files are written), run Nearest Neighbor on each, and show the cost, trucks used and run time.

The time column covers only the solver, not generating the instance.

In [6]:
import time
from generator.gen import generate

tests = [(50, 1), (200, 1), (500, 1), (1000, 1)]   # (customers, seed)
methods = [nearest_neighbor, clarke_wright]

print('%6s %9s %17s %8s %12s %9s %8s' % ('n', 'vehicles', 'method', 'trucks', 'cost', 'feasible', 'time'))
for n, seed in tests:
    demands, dist, capacity, vehicles = generate(n, seed, demand_max=10, route_size=10, slack=1.2)

    costs = {}
    for method in methods:
        start = time.perf_counter()
        routes = method(demands, dist, capacity)
        elapsed = time.perf_counter() - start

        errors, cost = validate(demands, dist, capacity, vehicles, routes)
        costs[method.__name__] = cost
        print('%6d %9d %17s %8d %12.1f %9s %7.2fs' % (n, vehicles, method.__name__, len(routes), cost, not errors, elapsed))

    gain = 100 * (costs['nearest_neighbor'] - costs['clarke_wright']) / costs['nearest_neighbor']
    print('%6s Clarke-Wright is %.1f%% cheaper than Nearest Neighbor\n' % ('', gain))

     n  vehicles            method   trucks         cost  feasible     time
    50         6  nearest_neighbor        5      12267.7      True    0.00s
    50         6     clarke_wright        6       8735.5      True    0.02s
       Clarke-Wright is 28.8% cheaper than Nearest Neighbor

   200        24  nearest_neighbor       21      30058.9      True    0.00s
   200        24     clarke_wright       21      23770.0      True    0.02s
       Clarke-Wright is 20.9% cheaper than Nearest Neighbor

   500        60  nearest_neighbor       50      63019.0      True    0.02s
   500        60     clarke_wright       51      51358.3      True    0.19s
       Clarke-Wright is 18.5% cheaper than Nearest Neighbor

  1000       122  nearest_neighbor      101     115832.1      True    0.10s
  1000       122     clarke_wright      102      94671.5      True    1.04s
       Clarke-Wright is 18.3% cheaper than Nearest Neighbor

